# Program 6: Compare responses to the original and the enriched prompt

**Aim.** Generate responses with an AI model for both prompts and compare detail and relevance.

**Theory.** Detail is measured as length and as coverage of a fixed checklist of expected topics; relevance as cosine similarity of the response to a reference answer using average GloVe vectors. Both are crude proxies and the run is one sample at temperature 0.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import json, re, numpy as np
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-100')
llm, label = get_llm()
print('model:', label)
P = json.loads((OUT / 'lab05_prompts.json').read_text(encoding='utf-8'))
r_orig = ask(llm, P['original'] + ' Answer in under 200 words.')
r_enr = ask(llm, P['enriched'] + ' Answer in under 200 words.')
print('--- ORIGINAL ---\n', r_orig, '\n--- ENRICHED ---\n', r_enr)

model: Groq openai/gpt-oss-20b (no Cohere key available)


--- ORIGINAL ---
 **How to Take Your New Medicine**

1. **Read the label and instructions** – Check the drug name, dose, frequency, and any special warnings.  
2. **Follow the prescription** – Take exactly the amount and at the times your doctor or pharmacist told you.  
3. **Timing** –  
   * If it’s a daily pill, take it at the same time each day.  
   * If it’s “with food” or “on an empty stomach,” follow that cue.  
4. **Use the right method** – Swallow whole with water unless the label says otherwise (e.g., chewable, dissolve in liquid).  
5. **Keep a schedule** – Use a pillbox, phone reminder, or calendar to avoid missed doses.  
6. **Store properly** – Keep it in a cool, dry place, out of reach of children.  
7. **Watch for side effects** – Mild nausea, dizziness, or rash can occur; note any new symptoms.  
8. **When to call your doctor** – If you miss a dose, feel worse, or notice serious side effects (difficulty breathing, swelling, severe rash).  
9. **Finish the course** – E

In [3]:
CHECK = {'dose': ['dose', 'dosage', 'amount'], 'timing': ['time', 'daily', 'morning', 'hours'], 'food': ['food', 'meal', 'empty stomach'],
         'side effects': ['side effect', 'reaction'], 'pharmacist or doctor': ['pharmacist', 'doctor'], 'storage': ['store', 'storage', 'cool', 'dry'],
         'missed dose': ['missed', 'forget'], 'allergy': ['allerg']}
def coverage(t): return [k for k, ws in CHECK.items() if any(w in t.lower() for w in ws)]
def avgvec(t):
    v = [wv[w] for w in re.findall(r'[a-z]+', t.lower()) if w in wv]
    return np.mean(v, axis=0)
ref = 'take the dose at the right time with food, watch for side effects, ask the pharmacist or doctor, store in a cool dry place, do not double a missed dose, tell about allergy'
cos = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
for name, r in (('original', r_orig), ('enriched', r_enr)):
    cv = coverage(r)
    print(f'{name:9s} words {len(r.split()):4d}  checklist {len(cv)}/{len(CHECK)}  cosine to reference {cos(avgvec(r), avgvec(ref)):.3f}  missing: {[k for k in CHECK if k not in cv]}')

original  words  201  checklist 7/8  cosine to reference 0.987  missing: ['allergy']
enriched  words  206  checklist 3/8  cosine to reference 0.978  missing: ['timing', 'food', 'pharmacist or doctor', 'missed dose', 'allergy']


## Observations

In this run the enriched prompt covered fewer checklist topics than the plain one (see the table). The retrieved words (veterinary, dentist, internist) pulled the answer towards other professions instead of how to take a medicine. Nearest neighbours in embedding space are related words, not necessarily useful ones, so enrichment needs a relevance filter and has to be measured over many prompts, not assumed to help. This is one sample at temperature 0.